In [1]:
# --- Standard Libraries ---
import os
import re
from collections import Counter

# --- Data Science ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# --- Utils ---
from tqdm.auto import tqdm
from IPython.display import display

# --- Setup ---
tqdm.pandas()


final_daily_df = pd.read_csv(
    os.path.join("processed", "final_daily_df.csv"),
    parse_dates=["date"]
)

print(final_daily_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 59 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3756 non-null   datetime64[ns]
 1   tweet_count               3756 non-null   int64         
 2   neg                       3001 non-null   float64       
 3   neu                       3001 non-null   float64       
 4   pos                       3001 non-null   float64       
 5   nlp_tweet_count           3001 non-null   float64       
 6   not_polarized             3001 non-null   float64       
 7   polarized                 3001 non-null   float64       
 8   anger                     3001 non-null   float64       
 9   disgust                   3001 non-null   float64       
 10  fear                      3001 non-null   float64       
 11  joy                       3001 non-null   float64       
 12  neutral             

# Vizualization

Visualize (daily and rolling window):
- Tweet activity over time
- Sentiment development over time
- Polarisation percentage over time
- Ekman Emotions over time
- Big 5 Personality traits over time
- Word distribution over time
- Tweet topics over time

In [6]:
# 1A: Tweet activity over time
fig1 = px.line(
    final_daily_df,
    x="date",
    y=["tweet_count", "nlp_tweet_count"],
    labels={"value": "Tweet-Anzahl", "variable": "Typ"},
    title="Elon Musks Tweet-Aktivität über Zeit (interaktiv)"
)

# 1B: Daily sentiment (pos/neu/neg)
fig2 = px.line(
    final_daily_df,
    x="date",
    y=["pos", "neu", "neg"],
    labels={"value": "Sentiment-Wahrscheinlichkeit", "variable": "Sentiment"},
    title="Durchschnittliches Sentiment pro Tag (interaktiv)"
)

# 1C: Polarization vs Neutral share per day
fig3 = px.line(
    final_daily_df,
    x="date",
    y=["polarized", "not_polarized"],
    labels={"value": "Anteil", "variable": "Kategorie"},
    title="Anteil polarisiert vs. neutral pro Tag (interaktiv)"
)

# Display the three Plotly figures
fig1.show()
fig2.show()
fig3.show()


# ---------------------------------------------------------------------------------
# 2) ROLLING-WINDOW (365-day) PLOTS (Plotly Graph Objects)
# ---------------------------------------------------------------------------------

window_size = 365  # 1-year rolling window

# 2A: Rolling tweet_count, not_polarized, polarized
rolling_meta = (
    final_daily_df[["date", "tweet_count", "not_polarized", "polarized"]]
    .set_index("date")
    .rolling(window=window_size, min_periods=1)
    .mean()
    .reset_index()
)
# Now rolling_meta has columns: date, tweet_count, not_polarized, polarized (all length = 3756)
rolling_meta["share_polarized"] = (
    rolling_meta["polarized"] 
    / (rolling_meta["polarized"] + rolling_meta["not_polarized"])
)
rolling_meta["share_neutral"] = (
    rolling_meta["not_polarized"] 
    / (rolling_meta["polarized"] + rolling_meta["not_polarized"])
)

# 2B: Rolling sentiment (pos, neu, neg)
rolling_sentiment = (
    final_daily_df[["date", "pos", "neu", "neg"]]
    .set_index("date")
    .rolling(window=window_size, min_periods=1)
    .mean()
    .reset_index()
)

# 2C: Rolling emotions
rolling_emotions = (
    final_daily_df[["date", "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]]
    .set_index("date")
    .rolling(window=window_size, min_periods=1)
    .mean()
    .reset_index()
)

# 2D: Rolling personality
rolling_personality = (
    final_daily_df[["date", "Extroversion", "Neuroticism", "Agreeableness", "Conscientiousness", "Openness"]]
    .set_index("date")
    .rolling(window=window_size, min_periods=1)
    .mean()
    .reset_index()
)

# -----------------------------------------------
# 2A: Rolling Tweet Activity Chart
# -----------------------------------------------
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=rolling_meta["date"],
    y=rolling_meta["tweet_count"],
    mode="lines",
    name="Rolling Tweet Count"
))
fig4.add_vline(
    x=pd.to_datetime("2022-10-27"),
    line=dict(color="red", dash="dash"),
    annotation_text="Twitter-Übernahme",
    annotation_position="top left"
)
fig4.update_layout(
    title="Rolling Tweet Count (365-Tage-Fenster)",
    xaxis_title="Datum",
    yaxis_title="Tweet-Anzahl"
)


# -----------------------------------------------
# 2B: Rolling Sentiment Trends Chart
# -----------------------------------------------
fig5 = go.Figure()
for sentiment_col in ["pos", "neu", "neg"]:
    fig5.add_trace(go.Scatter(
        x=rolling_sentiment["date"],
        y=rolling_sentiment[sentiment_col],
        mode="lines",
        name=sentiment_col.capitalize()
    ))
fig5.add_vline(
    x=pd.to_datetime("2022-10-27"),
    line=dict(color="red", dash="dash"),
    annotation_text="Twitter-Übernahme",
    annotation_position="top left"
)
fig5.update_layout(
    title="Rolling Sentiment Trends (365-Tage-Fenster)",
    xaxis_title="Datum",
    yaxis_title="Anteil (0–1)"
)


# -----------------------------------------------
# 2C: Rolling Polarized vs Neutral Share Chart
# -----------------------------------------------
fig6 = go.Figure()
fig6.add_trace(go.Scatter(
    x=rolling_meta["date"],
    y=rolling_meta["share_polarized"],
    mode="lines",
    name="Polarisiert"
))
fig6.add_trace(go.Scatter(
    x=rolling_meta["date"],
    y=rolling_meta["share_neutral"],
    mode="lines",
    name="Neutral"
))
fig6.update_layout(
    title="Rolling Anteil polarisiert vs. neutral (365-Tage-Fenster)",
    xaxis_title="Datum",
    yaxis_title="Anteil"
)


# -----------------------------------------------
# 2D: Rolling Emotion Scores Chart
# -----------------------------------------------
fig7 = go.Figure()
for emo_col in ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]:
    fig7.add_trace(go.Scatter(
        x=rolling_emotions["date"],
        y=rolling_emotions[emo_col],
        mode="lines",
        name=emo_col.capitalize()
    ))
fig7.update_layout(
    title="Rolling Emotion Scores (365-Tage-Fenster)",
    xaxis_title="Datum",
    yaxis_title="Score"
)


# -----------------------------------------------
# 2E: Rolling Big Five Personality Traits Chart
# -----------------------------------------------
fig8 = go.Figure()
for trait in ["Extroversion", "Neuroticism", "Agreeableness", "Conscientiousness", "Openness"]:
    fig8.add_trace(go.Scatter(
        x=rolling_personality["date"],
        y=rolling_personality[trait],
        mode="lines",
        name=trait
    ))
fig8.update_layout(
    title="Rolling Big Five Traits (365-Tage-Fenster)",
    xaxis_title="Datum",
    yaxis_title="Score (0–1)"
)


# Display all rolling-window figures
fig4.show()
fig5.show()
fig6.show()
fig7.show()
fig8.show()


# ---------------------------------------------------------------------------------
# 3) STATIC BARPLOT: Top 50 Words (Matplotlib/Seaborn)
# ---------------------------------------------------------------------------------

# Ensure `word_counts` is sorted descending by “count”
top_n = 50
plt.figure(figsize=(12, 10))
sns.barplot(
    data=word_counts.head(top_n),
    x="count",
    y="word",
    palette="viridis"
)
plt.title("Top 50 Words in Musk's Tweets")
plt.xlabel("Count")
plt.ylabel("Word")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------------
# 4) HEATMAP: Daily Share of Top Words
# ---------------------------------------------------------------------------------

# Extract the top‐N words overall
top_words = word_counts["word"].head(top_n).tolist()

# Filter the daily word‐counts matrix to only these top words
daily_top_matrix = word_counts_daily[top_words]  # shape = (num_dates × top_n)

# For visualization, normalize each row so that the sum across top_words = 1
daily_top_share = daily_top_matrix.div(daily_top_matrix.sum(axis=1), axis=0)

plt.figure(figsize=(14, 10))
sns.heatmap(
    daily_top_share,
    cmap="viridis",
    cbar_kws={"label": "Share of Top Words per Day"}
)
plt.title(f"Daily Share of Top {top_n} Words")
plt.xlabel("Word")
plt.ylabel("Date")
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------------
# 5) STACKED AREA CHART: Tweet Topics Over Time
# ---------------------------------------------------------------------------------

# Normalize each day's topic counts to proportions
topic_props = topic_daily.div(topic_daily.sum(axis=1), axis=0)

plt.figure(figsize=(14, 6))
dates = topic_props.index

plt.stackplot(
    dates,
    [topic_props[col] for col in topic_props.columns],
    labels=topic_props.columns
)
plt.legend(
    bbox_to_anchor=(1.04, 0.5),
    loc="center left",
    fontsize="small",
    ncol=1
)
plt.title("Daily Topic Distribution Over Time")
plt.xlabel("Date")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`

In [ ]:
fig1 = px.line(
    final_daily_df,
    x="date",
    y=["tweet_count", "nlp_tweet_count"],
    labels={"value": "Tweetanzahl", "variable": "Typ"},
    title="Elon Musks Tweet-Aktivität über Zeit (interaktiv)"
)

fig2 = px.line(
    final_daily_df,
    x="date",
    y=["pos", "neu", "neg"],
    labels={"value": "Sentiment-Wahrscheinlichkeit", "variable": "Sentiment"},
    title="Durchschnittliches Sentiment pro Tag (interaktiv)"
)

fig3 = px.line(
    final_daily_df,
    x="date",
    y=["polarized", "not_polarized"],
    labels={"value": "Anteil", "variable": "Kategorie"},
    title="Anteil polarisiert vs. neutral pro Tag (interaktiv)"
)

fig1.show()
fig2.show()
fig3.show()

window_size = 365

rolling_sentiment = final_daily_df[['pos', 'neu', 'neg']].rolling(window=window_size, min_periods=1).mean()
rolling_meta = final_daily_df[['tweet_count', 'not_polarized', 'polarized']].rolling(window=window_size, min_periods=1).mean()
rolling_emotions = final_daily_df[['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']].rolling(window=window_size, min_periods=1).mean()
rolling_emotions["date"] = final_daily_df["date"]
rolling_personality = final_daily_df[['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']].rolling(window=window_size, min_periods=1).mean()
rolling_personality["date"] = final_daily_df["date"]

# Plot 1: Tweet-Aktivität
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=final_daily_df["date"], y=rolling_meta['tweet_count'], mode='lines', name='Tweet Count (Rolling)'))
fig1.add_vline(x="2022-10-27", line=dict(color="red", dash="dash"), name="Twitter-Übernahme")
fig1.update_layout(title="Rolling Tweet Count", xaxis_title="Datum", yaxis_title="Tweetanzahl")

# Plot 2: Sentiment-Trends
fig2 = go.Figure()
for col in ['pos', 'neu', 'neg']:
    fig2.add_trace(go.Scatter(x=final_daily_df["date"], y=rolling_sentiment[col], mode='lines', name=col.capitalize()))
fig2.add_vline(x="2022-10-27", line=dict(color="red", dash="dash"))
fig2.update_layout(title="Rolling Sentiment Trends", xaxis_title="Datum", yaxis_title="Anteil (0–1)")

# Plot 3: Polarisiert vs Neutral
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=final_daily_df["date"], y=rolling_meta['polarized'], name='Polarisiert', mode='lines'))
fig3.add_trace(go.Scatter(x=final_daily_df["date"], y=rolling_meta['not_polarized'], name='Neutral', mode='lines'))
fig3.update_layout(title="Rolling Anteil polarisiert vs. neutral", xaxis_title="Datum", yaxis_title="Anteil")

# Plot 4: Emotionen
fig4 = go.Figure()
for col in ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']:
    fig4.add_trace(go.Scatter(x=rolling_emotions["date"], y=rolling_emotions[col], mode='lines', name=col.capitalize()))
fig4.update_layout(title="Rolling Emotion Scores", xaxis_title="Datum", yaxis_title="Score")

# Plot 5: Persönlichkeit
fig5 = go.Figure()
for col in ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']:
    fig5.add_trace(go.Scatter(x=rolling_personality["date"], y=rolling_personality[col], mode='lines', name=col))
fig5.update_layout(title="Rolling Big Five Traits", xaxis_title="Datum", yaxis_title="Score (0–1)")

fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()

# vizulaize Top 50 words
#plt.figure(figsize=(14, 100))
#sns.barplot(data=word_counts.head(100), x="count", y="word", palette="viridis")
#plt.title("Top 50 Words in Musk's Tweets")
#plt.xlabel("Count")
#plt.ylabel("Word")

# Vizalize top word daily

# Vizualize topics

DataError: Cannot aggregate non-numeric type: datetime64[ns]